# Bimodal: KL divergence of MGD vs REG (paper figure)

KL analysis of `analyse_bm_results.ipynb` (June 2026, `KL_bimodal_paper.png`) on a sweep
launched with `launch/bimodal_sweep.sh`. Reads, per beta dir, `experiment_K_runs.pt`
(raw MGD `theta_final_mgd`), `lam_selection.pt` (old E[z^2] rule, `select_lambda.py`) and
`lam_selection_A.pt` (rule A, `select_lambda_A.py`, 2026-11-19: the final rule). REG is taken at
the rule-A lam by default (`LAM = 'A'`).

* **KL vs ensemble mean** (the paper figure): `E_k[ KL(p_theta_bar || p_theta_k) ]`,
  `theta_bar` = mean over the K runs of the same estimator. A spread measure: bias is
  not counted.
* **KL vs truth**: `E_k[ KL(p_true || p_theta_k) ]`, includes the bias.
* Cramér–Rao: `r / (2 n1)` (to second order `KL ~ 1/2 dtheta^T Cov(phi) dtheta`).

theta are coefficients of `phi = [x, x^2/2, x^3/3, x^4/4]`; `p_theta ∝ exp(theta · phi)`.

In [11]:
import os
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt
from scipy.integrate import quad, simpson

from bimodal_utils import integrand     # the sampled density, for the convention check

# ── parameters ────────────────────────────────────────────────────────────
RESULTS = Path('results_bim_theta_beta')
# default: the newest sweep that has lam_selection.pt files
_sweeps = sorted(d for d in RESULTS.glob('bim_theta_beta_sweep_*') if any(d.glob('beta_*/lam_selection.pt')))
SWEEP_DIR = Path(os.environ.get('BM_SWEEP_DIR', _sweeps[-1] if _sweeps else ''))
LAM = 'A'        # 'A' = rule A (lam_selection_A.pt, final); 'selected' = old E[z^2] rule;
                 # 'oracle' = old file's oracle; or a value of the old lam grid
BETAS = None  # None = all
X_RANGE, N_GRID = (-10.0, 10.0), 8001

sel, mgd_final = [], []
for d in sorted(p for p in SWEEP_DIR.glob('beta_*') if p.is_dir()):
    if not (d / 'lam_selection.pt').exists():
        print('skipping', d.name, '(no lam_selection.pt)'); continue
    s = torch.load(d / 'lam_selection.pt', map_location='cpu', weights_only=False)
    if BETAS is not None and not np.isclose(BETAS, s['config']['beta']).any():
        continue
    e = torch.load(d / 'experiment_K_runs.pt', map_location='cpu', weights_only=False)
    # select_lambda.py solves the runs in the order of e['runs'] -> rows match
    assert e['theta_final_mgd'].shape[0] == s['theta_final_reg'].shape[0]
    sel.append(s); mgd_final.append(e['theta_final_mgd'].double().numpy())
betas = np.array([s['config']['beta'] for s in sel])
# rule A (cross-dataset CV, M metric, t >= 0.99), same runs in the same order as lam_selection.pt
selA = [torch.load(SWEEP_DIR / f'beta_{b:.4f}' / 'lam_selection_A.pt', map_location='cpu', weights_only=False)
        for b in betas]
for a, m in zip(selA, mgd_final):
    assert a['theta_final_reg'].shape[0] == len(m)
n1 = sel[0]['config']['n1']
print('SWEEP_DIR:', SWEEP_DIR, f'({len(sel)} betas)')

SWEEP_DIR: results_bim_theta_beta/bim_theta_beta_sweep_20260924_152418 (10 betas)


## How the KL is computed

All divergences are in **nats** (natural log). Everything below is a 1-D integral, done numerically on a grid; no samples and no histograms enter the KL.

### 1. The model family

With $r=4$ and the feature vector
$$\varphi(x)=\Big(x,\ \tfrac{x^2}{2},\ \tfrac{x^3}{3},\ \tfrac{x^4}{4}\Big),$$
every density in the family is fixed by a multiplier vector $\theta\in\mathbb R^4$:
$$p_\theta(x)=\frac{e^{\theta\cdot\varphi(x)}}{Z(\theta)},\qquad Z(\theta)=\int e^{\theta\cdot\varphi(x)}\,dx,\qquad \log p_\theta=\theta\cdot\varphi-\log Z(\theta).$$
$Z$ is finite only if $\theta_4<0$ (otherwise $p_\theta$ cannot be normalised; the code returns NaN for those rows).

### 2. Where each $\theta$ comes from

* **True multipliers $\theta^*$** (`target_theta_phi`): the multipliers of the density the data are drawn from, $p_{\rm true}\propto\exp(-\beta x^4+5\beta x^2+\tfrac{\beta}{2}x)$, i.e. $\theta^*=(\beta/2,\ 10\beta,\ 0,\ -4\beta)$. It is **known analytically, not estimated** (`bimodal_ensemble_reg.py`: coefficients of $x^a$, multiplied by $a$ to get the coefficients of $\varphi_a=x^a/a$). The check in the next code cell, $D_{\rm KL}(p_{\rm true}\Vert p_{\theta^*})\approx0$, only tests this change of basis and the grid code.
* **MGD estimates $\hat\theta_k^{\rm MGD}$, $k=1..K$** (`theta_final_mgd`, shape $(K,4)$): the multiplier vector at the end of the SDE (final time) of run $k$, with no time regularisation.
* **REG estimates $\hat\theta_k^{\rm REG}(\lambda)$** (`theta_final_reg[:, i]`): the same $K$ runs re-solved with the time-regularised system at strength $\lambda$. $\lambda$ is the one chosen by `select_lambda.py` (`lam_selected`, option `LAM='selected'`), or the oracle value, or a value of the grid.
* **Ensemble mean** $\bar\theta=\frac1K\sum_k\hat\theta_k$, taken **within the same estimator** (MGD with MGD, REG with REG).

REG is therefore *the same formulas with $\Theta_{\rm reg}$ in place of $\theta$*: both estimators are just a vector $\hat\theta$ plugged into $p_{\hat\theta}$.

### 3. The divergence

For two members of the family,
$$D_{\rm KL}(p_a\Vert p_b)=\int p_a\,\log\frac{p_a}{p_b}\,dx=(\theta_a-\theta_b)\cdot\langle\varphi\rangle_{p_a}-\log Z(\theta_a)+\log Z(\theta_b).$$
The first argument is the **reference** (the law the expectation is taken under). The code does not use the second form; it integrates the first form directly on a grid.

### 4. Numerics (`log_density`, `kl`)

* Grid $x_j$: `N_GRID = 8001` points on `X_RANGE = [-10, 10]`; integrals by Simpson's rule. This assumes $p_\theta$ has negligible mass outside $[-10,10]$.
* $\log Z$ by a stabilised log-sum-exp: with $\ell_j=\theta\cdot\varphi(x_j)$ and $m=\max_j\ell_j$,
$$\log Z=m+\log\!\int e^{\ell(x)-m}dx,$$
so that $e^{\ell}$ never overflows.
* $D_{\rm KL}(p_{\rm ref}\Vert p_\theta)\approx\sum_j w_j\,p_{\rm ref}(x_j)\,[\log p_{\rm ref}(x_j)-\log p_\theta(x_j)]$, with Simpson weights $w_j$ and the convention $0\log0=0$ where $p_{\rm ref}=0$.
* Rows with $\theta_4\ge0$ give NaN; they are skipped by `nanmean` and counted in the `NaN` column of the table.

### 5. The two quantities reported

For an estimator with runs $\hat\theta_1..\hat\theta_K$:

* **KL vs ensemble mean** (the paper figure): $\ \mathbb E_k\big[D_{\rm KL}(p_{\bar\theta}\Vert p_{\hat\theta_k})\big]$. This measures the **spread** of the runs around their own mean; a bias common to all runs is not counted. Line = mean over $k$, band = 25th–75th percentile over $k$.
* **KL vs truth**: $\ \mathbb E_k\big[D_{\rm KL}(p_{\theta^*}\Vert p_{\hat\theta_k})\big]$. This includes the bias.

### 6. The reference lines (second order)

Since $\nabla^2_\theta\log Z=\mathrm{Cov}_{p_\theta}(\varphi)=:I(\theta)$ (the Fisher information of the family),
$$D_{\rm KL}(p_\theta\Vert p_{\theta+\delta})=\tfrac12\,\delta^{\!\top} I(\theta)\,\delta+O(\|\delta\|^3).$$

* **`1/2 VarE` column**: $\frac12\cdot\frac{1}{K-1}\sum_k\delta_k^\top I\,\delta_k$ with $\delta_k=\hat\theta_k-\bar\theta$, i.e. $\frac12\mathrm{tr}(I\,\widehat{\mathrm{Cov}}(\hat\theta))$. Here $I$ is `cov_phi`, the sample covariance of $\varphi(x)$ over the reference sample of the experiment (`bimodal_ensemble_reg.py`). It should be close to the exact "KL vs mean" when the runs are close to $\bar\theta$; it is a check of the grid computation.
* **Cramér–Rao line** `cr` $=r/(2n_1)$ (`n1` from the config, $r=4$): for an **unbiased** estimator from $n_1$ i.i.d. samples, $\mathrm{Cov}(\hat\theta)\succeq I^{-1}/n_1$, so $\frac12\mathrm{tr}(I\,\mathrm{Cov}(\hat\theta))\ge r/(2n_1)$.
  The bound does **not** apply to a biased estimator: REG shrinks $\theta$ and may lie below this line in "KL vs ensemble mean", which ignores bias. The grey band "below the bound: unreachable" in the paper figure is only valid for unbiased estimators.


In [12]:
GRID = np.linspace(*X_RANGE, N_GRID)
PHI = np.stack([GRID, GRID**2 / 2, GRID**3 / 3, GRID**4 / 4], axis=1)      # (n_grid, 4)

def log_density(thetas):
    # normalised log p_theta on GRID, (M, n_grid); NaN rows where theta_4 >= 0 (not normalisable)
    thetas = np.atleast_2d(np.asarray(thetas, dtype=float))
    lv = thetas @ PHI.T
    m = lv.max(axis=1, keepdims=True)
    with np.errstate(over='ignore'):
        lv = lv - (m + np.log(simpson(np.exp(lv - m), x=GRID, axis=1))[:, None])
    lv[thetas[:, 3] >= 0] = np.nan
    return lv

def kl(log_p_ref, thetas, chunk=500):
    # KL(p_ref || p_theta) for every row of thetas; log_p_ref (n_grid,), may be -inf where p_ref = 0
    p = np.exp(log_p_ref)
    out = []
    for i in range(0, len(thetas), chunk):
        with np.errstate(invalid='ignore'):
            f = p * (log_p_ref - log_density(thetas[i:i + chunk]))
        out.append(simpson(np.where(p > 0, f, 0.0), x=GRID, axis=1))    # 0 log 0 := 0
    return np.concatenate(out)

# convention check: bimodal_utils.integrand (what x1 is drawn from) == p_theta* in this basis
print('KL(p_true || p_theta*)   (must be ~0)')
for s in sel:
    beta = s['config']['beta']
    Z, _ = quad(integrand, -np.inf, np.inf, args=(beta,))
    with np.errstate(divide='ignore'):
        log_p_true = np.log(integrand(GRID, beta)) - np.log(Z)
    print(f"  beta={beta:4.2f}: {kl(log_p_true, s['target_theta_phi'].numpy()[None])[0]: .2e}")

KL(p_true || p_theta*)   (must be ~0)
  beta=0.10: -3.82e-15
  beta=0.30:  2.61e-15
  beta=0.50: -3.40e-13
  beta=0.70:  1.35e-15
  beta=0.90:  2.57e-15
  beta=1.10:  3.76e-15
  beta=1.30:  1.25e-14
  beta=1.50: -1.78e-15
  beta=1.70:  9.83e-15
  beta=1.90: -5.86e-13


## KL per beta

In [13]:
def reg_of(s, a):
    """REG theta at t = 1 (K, 4), the lam used and 1/2 Var E (centered) for the LAM choice."""
    if LAM == 'A':
        i = a['lams'].index(a['lam_selected'])
        return (a['theta_final_reg'][:, i].double().numpy(), a['lam_selected'],
                0.5 * float(a['var_energy_reg_centered'][i]))
    lam = {'selected': s['lam_selected'], 'oracle': s['lam_oracle']}.get(LAM, LAM if not isinstance(LAM, str) else None)
    i = s['lams'].index(float(lam))
    return s['theta_final_reg'][:, i].numpy(), lam, 0.5 * float(s['var_energy_reg_centered'][i])

rows = []
for s, a, mgd in zip(sel, selA, mgd_final):
    reg, lam, quad_reg = reg_of(s, a)
    log_p_true = log_density(s['target_theta_phi'].numpy())[0]
    r = dict(beta=s['config']['beta'], K=len(mgd), lam=lam,
             # second-order check: KL_mean ~ 1/2 centered energy variance (select_lambda.py)
             quad_mgd=0.5 * float(s['var_energy_mgd_centered']),
             quad_reg=quad_reg)
    for name, th in [('mgd', mgd), ('reg', reg)]:
        r[f'{name}_mean'] = kl(log_density(th.mean(0))[0], th)
        r[f'{name}_true'] = kl(log_p_true, th)
    rows.append(r)

cr = sel[0]['theta_final_reg'].shape[-1] / (2 * n1)
avg = lambda a: np.nanmean(a)
print(f"LAM = {LAM!r}   Cramér–Rao r/(2 n1) = {cr:.1e}")
print(f"{'beta':>5} {'K':>4} {'lam':>7} | {'KL vs mean: MGD':>15} {'REG':>9} {'gain':>6} | {'1/2 VarE: MGD':>13} {'REG':>9}"
      f" | {'KL vs true: MGD':>15} {'REG':>9} | NaN")
for r in rows:
    nan = sum(np.isnan(r[k]).sum() for k in ('mgd_mean', 'reg_mean', 'mgd_true', 'reg_true'))
    print(f"{r['beta']:5.2f} {r['K']:4d} {r['lam']:7.0e} | {avg(r['mgd_mean']):15.3e} {avg(r['reg_mean']):9.3e} "
          f"{avg(r['mgd_mean']) / avg(r['reg_mean']):6.0f} | {r['quad_mgd']:13.3e} {r['quad_reg']:9.3e} | "
          f"{avg(r['mgd_true']):15.3e} {avg(r['reg_true']):9.3e} | {nan}")

LAM = 'selected'   Cramér–Rao r/(2 n1) = 2.0e-06
 beta    K     lam | KL vs mean: MGD       REG   gain | 1/2 VarE: MGD       REG | KL vs true: MGD       REG | NaN
 0.10  100   1e-04 |       1.592e-02 1.282e-04    124 |     1.643e-02 1.293e-04 |       1.600e-02 1.286e-04 | 0
 0.30  100   1e-04 |       1.799e-02 1.072e-04    168 |     1.877e-02 1.081e-04 |       1.825e-02 1.089e-04 | 0
 0.50  100   3e-05 |       2.529e-02 1.192e-04    212 |     2.631e-02 1.203e-04 |       2.536e-02 1.195e-04 | 0
 0.70  100   3e-05 |       3.324e-02 1.339e-04    248 |     3.597e-02 1.353e-04 |       3.398e-02 1.370e-04 | 0
 0.90  100   1e-05 |       4.482e-02 1.949e-04    230 |     4.798e-02 2.017e-04 |       4.573e-02 3.323e-04 | 0
 1.10  100   3e-06 |       5.538e-02 3.647e-04    152 |     6.303e-02 4.027e-04 |       5.795e-02 1.545e-03 | 0
 1.30  100   3e-06 |       6.665e-02 3.895e-04    171 |     7.747e-02 4.552e-04 |       7.036e-02 2.230e-03 | 0
 1.50  100   1e-06 |       8.577e-02 7.667e-04    112

## Rule A on this sweep (2026-11-19, job 426369)

$\lambda$ re-selected with **rule A** (`select_lambda_A.py`, launcher `launch/select_lambda_A.sh`), in its **cross-dataset** form: the $K=100$ runs of one $\beta$ have independent datasets $x_1$ (seed $42+k$) and independent SDE noise, and run $a$'s $\Theta_a(\lambda)$ is scored with the data term of every other run $b$,
$$L_{ab}(\lambda)=\sum_{t_k\ge0.99} w_k\Big[\tfrac12\,\Theta_{a,k}(\lambda)^{\!\top}M_{b,k}\,\Theta_{a,k}(\lambda)-b_{b,k}^{\top}\Theta_{a,k}(\lambda)\Big],\qquad \lambda_A=\arg\min_\lambda\sum_a\sum_{b\neq a}L_{ab}(\lambda),$$
($M_b,b_b$: run $b$'s saved system, the same that the $\lambda=0$ solve inverts, $M\theta=b$; $w_k$ trapezoid weights in $t$; ridge 0 as the solve). In the turbulence runs the replicas share $x_1$ ("cross-seed"); here they do not, so the data draw adds noise of order the Cramér–Rao level, negligible against the MGD spread. Grid: 0 and $10^{-10}\dots1$ in half decades (22 values). No new SDE run: the September `reg_system/` files were re-solved.

| $\beta$ | $\lambda$ old E[$z^2$] rule | $\lambda_A$ | $\lambda$ oracle (energy MSE vs $\theta^*$) | jackknife (leave one run out) | KL vs mean, REG at $\lambda_A$ | KL vs truth, REG at $\lambda_A$ | KL vs truth, REG at oracle |
|---|---|---|---|---|---|---|---|
| 0.1 | 1e-4 | 3e-4 | 3e-5 | 3e-4 (all 100) | 1.37e-4 | 1.40e-4 | 1.25e-4 |
| 0.3 | 1e-4 | 1e-4 | 3e-5 | 1e-4 | 1.07e-4 | 1.09e-4 | 1.04e-4 |
| 0.5 | 3e-5 | 3e-5 | 3e-5 | 3e-5 | 1.19e-4 | 1.20e-4 | 1.20e-4 |
| 0.7 | 3e-5 | 3e-5 | 3e-5 | 3e-5 | 1.34e-4 | 1.37e-4 | 1.37e-4 |
| 0.9 | 1e-5 | 1e-5 | 3e-5 | 1e-5 | 1.95e-4 | 3.32e-4 | 1.88e-4 |
| 1.1 | 3e-6 | 1e-5 | 3e-5 | 1e-5 | 2.97e-4 | 7.04e-4 | 6.05e-4 |
| 1.3 | 3e-6 | 3e-6 | 1e-5 | 3e-6 | 3.90e-4 | 2.23e-3 | 4.54e-4 |
| 1.5 | 1e-6 | 3e-6 | 3e-6 | 3e-6 | 7.46e-4 | 8.05e-4 | 8.05e-4 |
| 1.7 | 1e-6 | 1e-6 | 1e-7 | 1e-6 | 2.59e-3 | 1.58e-2 | 1.20e-2 |
| 1.9 | 1e-6 | 1e-6 | 1e-9 | 1e-6 | 1.74e-2 | 4.15e-1 | 3.62e-1 |

KL in nats, mean over the 100 runs (grid method of section "How the KL is computed"). The jackknife never moves $\lambda_A$; no $\lambda_A$ is at a grid edge; the pooled-sum shortcut matches the explicit pair sums to $\le 2\cdot10^{-15}$ (relative). Rule A differs from the old rule only at $\beta=0.1$, 1.1, 1.5 (one grid step each, always to the larger λ). At $\beta\ge1.7$ the KL to the truth is dominated by a bias that no $\lambda$ of the grid removes (REG at the oracle $\lambda$ is still $1.2\cdot10^{-2}$ and $0.36$ nats).


## Paper figure (KL vs ensemble mean)

In [ ]:
%matplotlib inline
C_MGD, C_REG, INK, MUTED = '#2a78d6', '#eb6834', '#0b0b0b', '#52514e'   # validated categorical slots 1-2
RC = {'font.size': 16, 'axes.labelsize': 22, 'xtick.labelsize': 17, 'ytick.labelsize': 17,
      'axes.spines.top': False, 'axes.spines.right': False,
      'axes.edgecolor': MUTED, 'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.labelcolor': INK}


def kl_figure(key_suffix, ylabel, fname):
    """Mean over the K runs (line) and 25-75 % band, MGD vs REG, with the Cramér–Rao line."""
    with plt.rc_context(RC):
        fig, ax = plt.subplots(figsize=(7.5, 5.5))
        for key, col, lab, mk in [('mgd' + key_suffix, C_MGD, 'MGD', 'o'),
                                  ('reg' + key_suffix, C_REG, 'Regularized MGD', 's')]:
            vals = [r[key][np.isfinite(r[key])] for r in rows]
            lo, hi = np.array([np.percentile(v, [25, 75]) for v in vals]).T
            ax.fill_between(betas, lo, hi, color=col, alpha=0.15, lw=0)          # spread over the K runs
            ax.plot(betas, [v.mean() for v in vals], '-' + mk, color=col, lw=2.2, ms=9,
                    mfc='white', mec=col, mew=2, label=lab, zorder=3)
        ax.axhline(cr, color=MUTED, ls='--', lw=1.8, label='Cramér–Rao bound')
        ax.set_yscale('log'); ax.set_xlabel(r'$|\theta|$')        # x values are beta, label as in the paper
        ax.set_ylabel(ylabel, fontsize=19)
        ax.set_ylim(cr / 5, None)
        ax.tick_params(width=1.2, length=5)
        ax.grid(axis='y', which='major', color='0.85', lw=0.8); ax.set_axisbelow(True)
        # legend in one row above the axes: can never cover the data
        ax.legend(loc='lower center', bbox_to_anchor=(0.5, 1.02), ncol=3, frameon=False,
                  handlelength=1.8, columnspacing=1.2, handletextpad=0.5, fontsize=15)
        plt.tight_layout()
        plt.savefig(fname, bbox_inches='tight')     # bbox: keep the legend above the axes in the file
        plt.show()


# file name carries the lam rule, so the June/September figures are not overwritten
kl_figure('_mean', r'$\mathbb{E}\,[D_{\rm KL}(p_{\bar\theta}\,\Vert\, p_{\hat\theta})]$',
          f'KL_bimodal_paper_lam{LAM}.png')


## Same, vs the true density (bias included)

In [ ]:
kl_figure('_true', r'$\mathbb{E}\,[D_{\rm KL}(p_{\theta^*}\,\Vert\, p_{\hat\theta})]$',
          f'KL_bimodal_truth_lam{LAM}.png')
